# Step 9: Re-evaluate Model 1 checkpoints on a bigger, leak-free ORD eval set (Colab GPU)

The current headline result (baseline→variant 2 fine-tuning) is statistically significant on the
existing 300-record ORD eval set (McNemar p=0.0062), but n=300 leaves several secondary
comparisons underpowered (e.g. variant 2 vs variant 4/150k: p=0.06–0.85, could be real or noise).

`data/v2_ord_eval_targets_2000.json` (committed to the repo, built via `build_eval_targets_ord.py
--count 2000 --exclude-file ...`) is a **new 2000-record ORD sample, verified 0 overlap** with the
original 300-record eval set AND every training pool used so far (57k/147k/297k). This notebook
re-runs `run_reactiont5_topk.py` (`--num-beams 10`) for the baseline checkpoint and the two fine-tuned
checkpoints (variant 2, variant 4) on this bigger set, on Colab GPU (much faster than local CPU for
2000 x beam-search records).

**Before running:** in the notebook Settings panel (right sidebar) turn on **Internet** and **GPU
accelerator** (T4).

**Checkpoint upload required** (baseline is public on HF, no upload needed — only the two
fine-tuned ones):
- Variant 2 (57k): local folder `~/Documents/diploma/retro-planner-checkpoints/model1_reactant_v2/final`
  (~758MB)
- Variant 4 (150k, v2cfg): local folder (from the Kaggle download) `model1_reactant_ord150k_v2cfg/final`
  (~758MB) — confirm you have the *v2cfg* one specifically (the correct-hyperparameter run reported
  in RESULTS.md variant 4), not `model1_reactant_ord150k` without that suffix, which may be the
  earlier wrong-hyperparameter attempt.

Upload both folders to Google Drive (drag-and-drop on drive.google.com is more reliable than the
browser file picker for large folders — same as the other notebooks' cross-account resume flow),
then point `v2_checkpoint_path` / `v4_checkpoint_path` below at wherever they land.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

`data/v2_ord_eval_targets_2000.json` is committed to the repo, so the git clone above already has
it — no separate build/upload step needed for the eval set itself.

In [ ]:
import json

n = len(json.load(open("data/v2_ord_eval_targets_2000.json")))
print("Eval targets:", n)
assert n == 2000

In [ ]:
v2_checkpoint_path = "/content/drive/MyDrive/retro-planner-checkpoints/model1_reactant_v2/final"  # @param {type:"string"}
v4_checkpoint_path = "/content/drive/MyDrive/retro-planner-checkpoints/model1_reactant_ord150k_v2cfg/final"  # @param {type:"string"}

import os
for p in (v2_checkpoint_path, v4_checkpoint_path):
    assert os.path.isdir(p), f"Not found: {p} -- did you upload it to Drive and set the path above?"
print("Both checkpoints found.")

**Baseline** (public HF checkpoint, no upload needed — downloads automatically).

In [ ]:
!python scripts/models/run_reactiont5_topk.py \
    --input data/v2_ord_eval_targets_2000.json \
    --t5-model sagawa/ReactionT5v2-retrosynthesis \
    --num-beams 10 --device cuda \
    --output experiments/v2_model1_topk/baseline_ord2000_topk.json

In [ ]:
!python scripts/models/run_reactiont5_topk.py \
    --input data/v2_ord_eval_targets_2000.json \
    --t5-model "{v2_checkpoint_path}" \
    --num-beams 10 --device cuda \
    --output experiments/v2_model1_topk/v2_ord2000_topk.json

In [ ]:
!python scripts/models/run_reactiont5_topk.py \
    --input data/v2_ord_eval_targets_2000.json \
    --t5-model "{v4_checkpoint_path}" \
    --num-beams 10 --device cuda \
    --output experiments/v2_model1_topk/v4_150k_ord2000_topk.json

**When all three finish:** download the three `experiments/v2_model1_topk/*_ord2000_topk.json`
files (each run prints its own summary at the end — scroll up in this notebook's output, or open
the JSON files directly from the Colab file browser on the left). Bring them back to the local repo
(`experiments/v2_model1_topk/`) so the McNemar significance check and RESULTS.md update can run
the same way earlier comparisons in this project did.